# AnalystLab Africa – ML Internship: Week 1–2 Assignment
## Data Preprocessing & Exploratory Data Analysis
**Datasets:** Titanic | Iris  
**Batch B | June–August 2025**

## Part 1: Data Loading & Initial Inspection

In [ ]:
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
import seaborn as sns
from sklearn.preprocessing import StandardScaler, LabelEncoder
from sklearn.datasets import load_iris
import warnings
warnings.filterwarnings('ignore')

sns.set_theme(style='whitegrid', palette='Set2')
plt.rcParams['figure.dpi'] = 120

In [ ]:
# Load Titanic
titanic = sns.load_dataset('titanic')
print("TITANIC — Shape:", titanic.shape)
titanic.head(10)

In [ ]:
# Titanic info & missing values
print(titanic.info())
print("\nMissing values:")
print(titanic.isnull().sum())
print(f"\nDuplicates: {titanic.duplicated().sum()}")

In [ ]:
titanic.describe(include='all')

In [ ]:
# Load Iris
iris_raw = load_iris(as_frame=True)
iris = iris_raw.frame.rename(columns={'target': 'species'})
iris['species'] = iris['species'].map({0:'Iris-setosa',1:'Iris-versicolor',2:'Iris-virginica'})
iris.columns = ['sepal_length','sepal_width','petal_length','petal_width','species']
print("IRIS — Shape:", iris.shape)
iris.head(10)

In [ ]:
print(iris.info())
print("\nMissing values:", iris.isnull().sum().sum())
print(f"Duplicates: {iris.duplicated().sum()}")
iris.describe()

## Part 2: Data Preprocessing
### A. Missing Value Treatment

In [ ]:
t = titanic.copy()
print("Missing values before treatment:")
print(t.isnull().sum()[t.isnull().sum() > 0])

In [ ]:
# Age — fill with median (robust to outliers)
t['age'].fillna(t['age'].median(), inplace=True)
# Embarked — fill with mode (most common port)
t['embarked'].fillna(t['embarked'].mode()[0], inplace=True)
t['embark_town'].fillna(t['embark_town'].mode()[0], inplace=True)
# Deck — 77% missing, drop the column
t.drop(columns=['deck','alive','who','adult_male','alone','class'], inplace=True)
print("Missing values after treatment:", t.isnull().sum().sum())

### B. Duplicate Removal

In [ ]:
dups = t.duplicated().sum()
t.drop_duplicates(inplace=True)
print(f"Titanic duplicates removed: {dups}")
print(f"New shape: {t.shape}")

### C. Outlier Detection & Treatment

In [ ]:
fig, axes = plt.subplots(1, 2, figsize=(12, 5))
axes[0].boxplot(titanic['fare'].dropna(), patch_artist=True, boxprops=dict(facecolor='salmon'))
axes[0].set_title("Fare — Before Capping"); axes[0].set_ylabel("Fare (£)")
Q1, Q3 = t['fare'].quantile(0.25), t['fare'].quantile(0.75)
upper = Q3 + 1.5 * (Q3 - Q1)
outliers = (t['fare'] > upper).sum()
t['fare'] = np.where(t['fare'] > upper, upper, t['fare'])
axes[1].boxplot(t['fare'], patch_artist=True, boxprops=dict(facecolor='lightgreen'))
axes[1].set_title(f"Fare — After Capping at £{upper:.2f} ({outliers} outliers capped)")
plt.tight_layout(); plt.show()
print(f"Outliers capped: {outliers}")

### D. Categorical Variable Encoding

In [ ]:
# Label Encoding: sex
t['sex_enc'] = LabelEncoder().fit_transform(t['sex'])
print("Sex encoding: female=0, male=1")
print(t[['sex','sex_enc']].drop_duplicates())

# One-Hot Encoding: embarked
embarked_dummies = pd.get_dummies(t['embarked'], prefix='emb', drop_first=True)
t = pd.concat([t, embarked_dummies], axis=1)
t.drop(columns=['sex','embarked','embark_town'], inplace=True)
print("\nNew columns:", list(t.columns))

In [ ]:
# Iris encoding
ir = iris.copy()
ir.drop_duplicates(inplace=True)
ir['species_enc'] = LabelEncoder().fit_transform(ir['species'])
print("Iris species encoding:")
print(ir[['species','species_enc']].drop_duplicates())

### E. Feature Scaling

In [ ]:
num_cols_t = ['age','fare','parch','sibsp']
scaler = StandardScaler()
t_scaled = t.copy()
t_scaled[num_cols_t] = scaler.fit_transform(t[num_cols_t])
print("Titanic — scaled stats:")
print(t_scaled[num_cols_t].describe().round(3))

In [ ]:
num_cols_ir = ['sepal_length','sepal_width','petal_length','petal_width']
ir_scaled = ir.copy()
ir_scaled[num_cols_ir] = scaler.fit_transform(ir[num_cols_ir])
print("Iris — scaled stats:")
print(ir_scaled[num_cols_ir].describe().round(3))

## Part 3: EDA
### 3A. Univariate Analysis — Titanic

In [ ]:
fig, axes = plt.subplots(2, 3, figsize=(15, 9))
fig.suptitle("TITANIC — Univariate Analysis", fontsize=15, fontweight='bold')
axes[0,0].hist(t['age'], bins=30, color='steelblue', edgecolor='white')
axes[0,0].set_title("Age Distribution"); axes[0,0].set_xlabel("Age")
axes[0,1].hist(t['fare'], bins=30, color='salmon', edgecolor='white')
axes[0,1].set_title("Fare (capped)"); axes[0,1].set_xlabel("Fare £")
titanic['survived'].value_counts().plot(kind='bar', ax=axes[0,2], color=['#e74c3c','#2ecc71'], edgecolor='white')
axes[0,2].set_title("Survival Count"); axes[0,2].set_xticklabels(['Died','Survived'], rotation=0)
titanic['pclass'].value_counts().sort_index().plot(kind='bar', ax=axes[1,0], color='mediumpurple', edgecolor='white')
axes[1,0].set_title("Passenger Class"); axes[1,0].set_xticklabels(['1st','2nd','3rd'], rotation=0)
axes[1,1].boxplot(t['age'], patch_artist=True, boxprops=dict(facecolor='lightblue'))
axes[1,1].set_title("Age Boxplot")
titanic['sex'].value_counts().plot(kind='bar', ax=axes[1,2], color=['#3498db','#e91e63'], edgecolor='white')
axes[1,2].set_title("Sex"); axes[1,2].set_xticklabels(['Male','Female'], rotation=0)
plt.tight_layout(); plt.show()

### 3A. Univariate Analysis — Iris

In [ ]:
features = ['sepal_length','sepal_width','petal_length','petal_width']
fig, axes = plt.subplots(2, 4, figsize=(16, 8))
fig.suptitle("IRIS — Univariate Analysis", fontsize=15, fontweight='bold')
colors = sns.color_palette("Set2", 4)
for i, feat in enumerate(features):
    axes[0,i].hist(iris[feat], bins=20, color=colors[i], edgecolor='white')
    axes[0,i].set_title(f"{feat}\nHistogram")
    axes[1,i].boxplot([ir[ir['species']==s][feat].values for s in ir['species'].unique()],
                      labels=['setosa','versicolor','virginica'], patch_artist=True,
                      boxprops=dict(facecolor=colors[i], alpha=0.7))
    axes[1,i].set_title(f"{feat}\nby Species"); axes[1,i].tick_params(axis='x', rotation=20)
plt.tight_layout(); plt.show()

### 3B. Bivariate Analysis — Titanic

In [ ]:
fig, axes = plt.subplots(1, 3, figsize=(15, 5))
fig.suptitle("TITANIC — Bivariate Analysis", fontsize=15, fontweight='bold')
survival_sex = titanic.groupby('sex')['survived'].mean()
axes[0].bar(survival_sex.index, survival_sex.values, color=['#e91e63','#3498db'], width=0.5)
axes[0].set_title("Survival Rate by Gender"); axes[0].set_ylim(0,1)
for i, v in enumerate(survival_sex.values): axes[0].text(i, v+0.02, f"{v:.1%}", ha='center', fontweight='bold')
survival_class = titanic.groupby('pclass')['survived'].mean()
axes[1].bar(['1st','2nd','3rd'], survival_class.values, color=['gold','silver','#cd7f32'], width=0.5)
axes[1].set_title("Survival Rate by Class"); axes[1].set_ylim(0,1)
for i, v in enumerate(survival_class.values): axes[1].text(i, v+0.02, f"{v:.1%}", ha='center', fontweight='bold')
d0 = titanic[titanic['survived']==0]['age'].dropna()
d1 = titanic[titanic['survived']==1]['age'].dropna()
bp = axes[2].boxplot([d0, d1], labels=['Died','Survived'], patch_artist=True, medianprops=dict(color='black', linewidth=2))
bp['boxes'][0].set_facecolor('#e74c3c'); bp['boxes'][0].set_alpha(0.6)
bp['boxes'][1].set_facecolor('#2ecc71'); bp['boxes'][1].set_alpha(0.6)
axes[2].set_title("Age vs Survival")
plt.tight_layout(); plt.show()

### 3B. Bivariate Analysis — Iris

In [ ]:
palette = {'Iris-setosa':'#2ecc71','Iris-versicolor':'#3498db','Iris-virginica':'#e74c3c'}
fig, axes = plt.subplots(1, 3, figsize=(15, 5))
fig.suptitle("IRIS — Bivariate Analysis", fontsize=15, fontweight='bold')
sns.boxplot(data=ir, x='species', y='petal_length', ax=axes[0], palette=['#2ecc71','#3498db','#e74c3c'])
axes[0].set_title("Petal Length by Species"); axes[0].set_xticklabels(['Setosa','Versicolor','Virginica'], rotation=15)
sns.boxplot(data=ir, x='species', y='petal_width', ax=axes[1], palette=['#2ecc71','#3498db','#e74c3c'])
axes[1].set_title("Petal Width by Species"); axes[1].set_xticklabels(['Setosa','Versicolor','Virginica'], rotation=15)
for sp, col in palette.items():
    sub = ir[ir['species']==sp]
    axes[2].scatter(sub['sepal_length'], sub['sepal_width'], label=sp.replace('Iris-',''), color=col, alpha=0.8, s=50)
axes[2].set_title("Sepal Length vs Sepal Width"); axes[2].legend(fontsize=8)
plt.tight_layout(); plt.show()

## Part 4: Correlation Analysis

In [ ]:
fig, axes = plt.subplots(1, 2, figsize=(16, 6))
fig.suptitle("Part 4: Correlation Heatmaps", fontsize=15, fontweight='bold')
num_t = t[['survived','pclass','age','sibsp','parch','fare','sex_enc']].copy()
corr_t = num_t.corr()
sns.heatmap(corr_t, annot=True, fmt='.2f', cmap='coolwarm', ax=axes[0], linewidths=0.5, vmin=-1, vmax=1)
axes[0].set_title("Titanic Correlation Heatmap"); axes[0].tick_params(axis='x', rotation=30)
corr_ir = ir[['sepal_length','sepal_width','petal_length','petal_width','species_enc']].corr()
sns.heatmap(corr_ir, annot=True, fmt='.2f', cmap='YlOrRd', ax=axes[1], linewidths=0.5, vmin=-1, vmax=1)
axes[1].set_title("Iris Correlation Heatmap"); axes[1].tick_params(axis='x', rotation=30)
plt.tight_layout(); plt.show()

In [ ]:
print("TITANIC — Correlation with 'survived':")
print(corr_t['survived'].sort_values(ascending=False).to_string())
print("\nIRIS — Correlation with 'species_enc':")
print(corr_ir['species_enc'].sort_values(ascending=False).to_string())

## Part 5: Machine Learning Preparation Summary

In [ ]:
print("""
╔══════════════════════════════════════════════════════════════╗
║        MACHINE LEARNING PREPARATION SUMMARY                 ║
╠══════════════════════════════════════════════════════════════╣
║ TITANIC                                                      ║
║ • Missing values: age→median, embarked→mode, deck→dropped   ║
║ • Outliers: fare capped at IQR upper bound (£73.86)         ║
║ • Encoding: sex→LabelEnc, embarked→OneHot (emb_Q, emb_S)   ║
║ • Scaling: StandardScaler on age, fare, parch, sibsp        ║
║ • Top features: sex_enc, pclass, fare, age                  ║
╠══════════════════════════════════════════════════════════════╣
║ IRIS                                                         ║
║ • Missing values: None                                       ║
║ • Outliers: sepal_width outliers retained                    ║
║ • Encoding: species→LabelEnc (0/1/2)                        ║
║ • Scaling: StandardScaler on all 4 numerical features       ║
║ • Top features: petal_width, petal_length                   ║
╚══════════════════════════════════════════════════════════════╝
Both datasets are now ready for ML model development.
""")

In [ ]:
# Save cleaned datasets
t.to_csv('titanic_cleaned.csv', index=False)
ir.to_csv('iris_cleaned.csv', index=False)
print("Cleaned datasets saved: titanic_cleaned.csv | iris_cleaned.csv")

### Iris Pairplot (Bonus)

In [ ]:
palette = {'Iris-setosa':'#2ecc71','Iris-versicolor':'#3498db','Iris-virginica':'#e74c3c'}
g = sns.pairplot(ir, hue='species', palette=palette, plot_kws={'alpha':0.7, 's':40})
g.figure.suptitle("Iris – All Feature Combinations", y=1.02, fontsize=13, fontweight='bold')
plt.show()